In [2]:
import numpy as np
import matplotlib.pyplot as plt

In [3]:
#  === 
# INPUTS
#  === 

x_lims = (369702.8, 374398.9)
y_lims = (407778.9, 411690.2)
z_lims = (0., -2500)

resolution = [100, 100, 60]   # nx, ny, nz

aperture = 50 #[m]
radius = 1000 #[m]
density = 1e-10 #[1/m3]
azimuth = 45 #[deg]

#  === 

x = np.linspace(*x_lims, resolution[0])
y = np.linspace(*y_lims, resolution[1])
z = np.linspace(*z_lims, resolution[2])

vol = np.abs(np.subtract(*x_lims) * np.subtract(*y_lims) * np.subtract(*z_lims)) #[m3]

azimuth = np.deg2rad(azimuth)

In [4]:
# A. Total number of fractures 
N = np.random.poisson(lam=vol*density, size=1)
N = 7
print('N =', N)

# B. Generate centers
x_0 = np.random.choice(x, size=N)
y_0 = np.random.choice(y, size=N)
z_0 = np.random.choice(z, size=N)

# C. Randomize apertures, radii and orientations for each fracture

# D. Define geometry


N = 7


In [5]:


# =========================
# GRID
# =========================
x = np.linspace(*x_lims, resolution[0])
y = np.linspace(*y_lims, resolution[1])
z = np.linspace(*z_lims, resolution[2])

X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

# =========================
# DIRECCIONES
# =========================

north = np.array([0.0, 1.0, 0.0])
east  = np.array([1.0, 0.0, 0.0])

u = np.cos(azimuth) * north + np.sin(azimuth) * east
v = -np.sin(azimuth) * north + np.cos(azimuth) * east
z_hat = np.array([0.0, 0.0, 1.0])

# =========================
# MASK
# =========================
mask = np.zeros(X.shape, dtype=bool)

# =========================
# LOOP OVER DISKS
# =========================
for i in range(N):

    C = np.array([x_0[i], y_0[i], z_0[i]])

    # vector desde centro a todos los puntos
    RX = X - C[0]
    RY = Y - C[1]
    RZ = Z - C[2]

    # eje alineado con azimuth
    s = RX * u[0] + RY * u[1] + RZ * u[2]

    # vertical
    t = RZ

    # perpendicular horizontal
    w = RX * v[0] + RY * v[1] + RZ * v[2]

    # disco en plano (u,z)
    inside_disk = (s**2 + t**2) <= radius**2

    # espesor perpendicular al azimuth
    along_azimuth = np.abs(w) <= aperture / 2

    mask |= (inside_disk & along_azimuth)

In [6]:
import plotly.graph_objects as go

# extraer voxels activos
ix, iy, iz = np.where(mask)

fig = go.Figure(data=[
    go.Scatter3d(
        x=x[ix],
        y=y[iy],
        z=z[iz],
        mode='markers',
        marker=dict(
            size=2,
            color=z[iz],
            colorscale='Viridis',
            opacity=0.6
        )
    )
])

fig.update_layout(
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='data'
    )
)

fig.show()